# Direct Steering

Most JuPedSim agents navigate autonomously via journeys.  **Direct steering** lets
you take over: `simulation.add_direct_steering_stage()` creates a special stage
where you set `agent.target` manually on every iteration, overriding the built-in
pathfinding.  The agent still obeys collision avoidance, but *you* decide where it
heads at each time step.

Typical use cases include scripted characters, remote-controlled agents, or
testing navigation geometry without relying on the journey system.


## Example – Driving an agent through a waypoint sequence

One agent is steered through three waypoints by setting `agent.target` each
iteration.  When the agent comes within 0.5 m of the current waypoint the
script advances to the next one; when the last waypoint is reached the agent
is removed.


In [ ]:
import pathlib

import jupedsim as jps
import shapely

trajectory_file = pathlib.Path("direct_steering.sqlite")

geometry = shapely.Polygon([(0, 0), (20, 0), (20, 10), (0, 10)])
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=trajectory_file, commit_every_nth_write=1
    ),
)

steering_stage = simulation.add_direct_steering_stage()
journey_id = simulation.add_journey(jps.JourneyDescription([steering_stage]))

agent_id = simulation.add_agent(
    jps.CollisionFreeSpeedModelAgentParameters(
        journey_id=journey_id,
        stage_id=steering_stage,
        position=(2, 5),
        radius=0.12,
    )
)

waypoints = [(10, 5), (10, 9), (18, 9)]
waypoint_index = 0
done = False

while (
    simulation.agent_count() > 0
    and simulation.iteration_count() < 10_000
    and not done
):
    agent = simulation.agent(agent_id)
    agent.target = waypoints[waypoint_index]
    if (
        shapely.Point(agent.position).distance(
            shapely.Point(waypoints[waypoint_index])
        )
        < 0.5
    ):
        waypoint_index += 1
        if waypoint_index >= len(waypoints):
            simulation.mark_agent_for_removal(agent_id)
            done = True
    simulation.iterate()

print(f"Done in {simulation.iteration_count()} iterations.")

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Add a second steered agent that follows a different path, or modify the
waypoint sequence for the existing agent:

```python
agent_id2 = simulation.add_agent(
    jps.CollisionFreeSpeedModelAgentParameters(
        journey_id=journey_id,
        stage_id=steering_stage,
        position=(2, 2),
        radius=0.12,
    )
)
waypoints2 = [(10, 2), (18, 2)]
waypoint_index2 = 0
```

Then inside the simulation loop, steer `agent_id2` through `waypoints2` with
the same proximity check as the first agent.


## See also

- {doc}`Journeys and Transitions </notebooks/fundamentals/07_journeys_transitions>` – journey-based navigation
- {doc}`Routing </notebooks/fundamentals/08_routing>` – runtime re-routing
- {doc}`../../concepts/object_model.rst </concepts/object_model>` – JuPedSim object model overview
- [JuPedSim scenario gallery](https://scenarios.jupedsim.org)
